# **7장 앙상블 학습과 랜덤 포레스트**

- **앙상블**: 일련의 예측기 - 분류나 회귀 모델의 예측을 모아 하나의 모델보다 더 좋은 예측을 만듬.
- **랜덤 포레스트**: 결정 트리의 앙상블

# **7.1 투표 기반 분류기**

**직접 투표**: 각 분류기의 예측을 집계하여 가장 많은 표를 얻은 클래스를 예측

- 개별 분류기 중 가장 뛰어난 분류기보다 정확도가 높은 경우가 많음.
- **약한 학습기**: 랜덤 추측보다 조금 더 높은 성능을 내는 분류기.

-> 약한 학습기가 충분히 많고 다양하면 높은 정확도를 내는 강한 학습기가 될 수 있음.

**큰 수의 법칙**

- 분류기에 적용하려면 분류기들이 독립적이고 오차에 상관관계가 없어야 함.
- 같은 데이터로 훈련하면 비슷한 오차를 만들기 쉬움.(잘못된 클래스가 다수->앙상블 정확도가 낮아짐) 서로 다른 알고리즘으로 학습시켜 다양한 분류기를 얻을 수 있음.

-> 앙상블 방법은 에측기가 가능한 서로 독립적일 때 최고성능(매우 다른 종류의 오차를 만들기 때)

**VotingClassifier**

- 이름/예측기 쌍의 리스트를 전달하여 투표 기반 분류기를 만듦.
- 훈련할 때 모든 추정기를 복제하고 복제된 추정기를 훈련함.
- 원본 추정기: `estimators`-for원본추정기, `named_estimators`-리스트대신 딕셔너리를 전달하는 경.
- 훈련된 복제본: `estimators_`-여기에 저장, `named_estimators_`.


In [23]:
#moons 데이터셋을 훈련 세트와 테스트 세트로 나누고, 로지스틱 회귀·랜덤 포레스트·SVC로 투표 기반 분류기를 훈련

from sklearn.datasets import make_moons
from sklearn.ensemble import RandomForestClassifier, VotingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.svm import SVC

X, y = make_moons(n_samples=500, noise=0.30, random_state=42)
X_train, X_test, y_train, y_test = train_test_split(X, y, random_state=42)

voting_clf = VotingClassifier(
    estimators=[
        ('lr', LogisticRegression(random_state=42)),
        ('rf', RandomForestClassifier(random_state=42)),
        ('svc', SVC(random_state=42))
    ]
)
voting_clf.fit(X_train, y_train)

VotingClassifier(estimators=[('lr', LogisticRegression(random_state=42)),
                             ('rf', RandomForestClassifier(random_state=42)),
                             ('svc', SVC(random_state=42))])

In [24]:
for name, clf in voting_clf.named_estimators_.items():
    print(name, "=", clf.score(X_test, y_test))

lr = 0.864
rf = 0.896
svc = 0.896


In [25]:
voting_clf.predict(X_test[:1])

array([1])

In [26]:
[clf.predict(X_test[:1]) for clf in voting_clf.estimators_]

[array([1]), array([1]), array([0])]

In [27]:
#테스트 세트에서 투표기반 분류기의 성능
voting_clf.score(X_test, y_test)
#결과가 투표기반 분류기가 다른 개별 분류기보다 더 높은 성능

0.912

**간접 투표**: 개별 분류기의 클래스 확률을 평균 내어 확률이 가장 높은 클래스를 예측

- 모든 분류기가 `predict_proba()`를 제공해야 함.
- 확률이 높은 투표에 비중을 더 둠.
- `voting="soft"`로 설정함.
- SVC는 `probability=True`로 설정해야 클래스 확률을 제공함. 확률 추정에 교차 검증을 사용하므로 훈련 속도가 느려짐.

In [28]:
voting_clf.voting = "soft"
voting_clf.named_estimators["svc"].probability = True
voting_clf.fit(X_train, y_train)
voting_clf.score(X_test, y_test)
#결과 해석->92%정확도

0.92

# **7.2 배깅과 페이스팅**

-> 같은 알고리즘을 사용하면서 훈련 세트의 서브셋을 랜덤으로 구성하여 예측기를 다르게 학습시키는 방법

- **배깅**: 중복을 허용하여 샘플링, bootstrap aggregating의 줄임말.
- **페이스팅**: 중복을 허용하지 않고 샘플링

-> 두 방법 모두 같은 훈련 샘플을 여러 예측기에 사용할 수 있지만, 한 예측기에 같은 샘플을 여러 번 사용하는 것은 배깅만 가능함.
- 예측을 모을 때 분류는 일반적으로 **통계적 최빈값**, 회귀는 **평균**을 사용함.
- 하나의 예측기와 비교하면 일반적으로 **편향은 비슷**하지만 **분산은 줄어듦**.
- 예측기들의 훈련과 예측을 병렬로 수행할 수 있음.

### **7.2.1 사이킷런의 배깅과 페이스팅**

- 분류: `BaggingClassifier`, 회귀: `BaggingRegressor`.
- `n_estimators=500`: 결정 트리 분류기 500개
- `max_samples=100`: 각 분류기를 랜덤으로 선택한 100개 샘플로 훈련
- `max_samples`를 0.0~1.0 사이의 실수로 지정하면 훈련 세트 크기에 그 값을 곱한 수만큼 샘플링
- `bootstrap=False`로 지정하면 페이스팅을 사용.
- `n_jobs=-1`: 가용한 모든 CPU 코어를 사용.
- 기반 분류기에 `predict_proba()`가 있으면 `BaggingClassifier`는 자동으로 간접 투표를 사용.

In [29]:
from sklearn.ensemble import BaggingClassifier
from sklearn.tree import DecisionTreeClassifier

bag_clf = BaggingClassifier(DecisionTreeClassifier(), n_estimators=500,
                            max_samples=100, n_jobs=-1, random_state=42)
bag_clf.fit(X_train, y_train)

BaggingClassifier(estimator=DecisionTreeClassifier(), max_samples=100,
                  n_estimators=500, n_jobs=-1, random_state=42)

**단일 결정 트리와 배깅 앙상블의 비교**

- 배깅 앙상블은 단일 결정 트리보다 결정 경계가 덜 불규칙하고 일반화가 더 잘됨.
- 배깅은 페이스팅보다 서브셋의 다양성이 높아 편향이 조금 더 높지만, 예측기 사이의 상관관계를 줄여 앙상블의 분산을 낮춤.

-> 시간과 CPU 파워에 여유가 있으면 교차 검증으로 두 방법을 비교하여 선택

### **7.2.2 OOB 평가**

- 배깅에서 한 예측기의 훈련에 선택되지 않은 샘플 = **OOB 샘플**
- 기본 설정에서는 중복을 허용하여 훈련 세트 크기만큼 샘플을 뽑음. 평균적으로 훈련 샘플의 약 63%가 선택되고 약 37%가 남음.
- 예측기마다 남겨진 OOB 샘플이 다름.
- OOB 샘플은 해당 예측기의 훈련에 사용되지 않으므로 별도의 검증 세트 없이 평가에 사용할 수 있음.

샘플 수 m이 커지면 선택되지 않을 확률은 $e^{-1}$에 가까워지고 선택되는 비율은 밑 값에 가까워짐.

$$
1-\exp(-1)\approx 63.212\%
$$

- `oob_score=True`: 훈련이 끝난 후 자동으로 OOB 평가를 수행
- `oob_score_`: OOB 평가 점수를 저장

In [30]:
bag_clf = BaggingClassifier(DecisionTreeClassifier(), n_estimators=500,
                            oob_score=True, n_jobs=-1, random_state=42)
bag_clf.fit(X_train, y_train)
bag_clf.oob_score_
#결과 - OOB 평가에서는 약 89.6%의 정확도를 예상

0.896

In [31]:
from sklearn.metrics import accuracy_score
y_pred = bag_clf.predict(X_test)
accuracy_score(y_test, y_pred)
#결과: `0.91200000000000003`. 테스트 세트의 정확도는 91.2% - 이 예제에서 OOB 평가는 조금 낮게 나타

0.92

**oob_decision_function_**

- OOB 샘플에 대한 결정 함수의 값을 확인함.
- 기반 예측기에 `predict_proba()`가 있으므로 각 훈련 샘플의 클래스 확률을 반환

In [32]:
bag_clf.oob_decision_function_[:3] # 처음 3개의 샘플에 대한 확률
#결과- 첫 번째 샘플이 양성 클래스에 속할 확률 - 67.6%, 음성 클래스에 속할 확률- 32.4%

array([[0.32352941, 0.67647059],
       [0.3375    , 0.6625    ],
       [1.        , 0.        ]])

# **7.3 랜덤 패치와 랜덤 서브스페이스**

`BaggingClassifier`는 샘플뿐 아니라 특성도 샘플링할 수 있음.

- `max_features`, `bootstrap_features`: 이 매개변수로 특성 샘플링을 조절함.
- 각 예측기는 랜덤으로 선택한 입력 특성의 일부로 훈련됨.
- 이미지와 같은 고차원 데이터셋에서 훈련 속도를 크게 높일 수 있음.

**랜덤 패치 방식**: 훈련 특성과 샘플을 모두 샘플링

**랜덤 서브스페이스 방식**: 훈련 샘플은 모두 사용하고 특성을 샘플링

- 샘플: `bootstrap=False`, `max_samples=1.0`
- 특성: `bootstrap_features=True` 그리고/또는 `max_features`를 1.0보다 작게 설정

-> 특성 샘플링은 예측기를 다양하게 만들어 편향을 늘리는 대신 분산을 낮춤.

# **7.4 랜덤 포레스트**

- 일반적으로 배깅 또는 페이스팅을 적용한 결정 트리의 앙상블
- 분류: `RandomForestClassifier`, 회귀: `RandomForestRegressor`
- 결정 트리의 성장과 앙상블을 제어하는 매개변수를 제공
- 노드를 분할할 때 전체 특성 대신 랜덤으로 선택한 특성 후보 중에서 최적의 특성을 찾음.
- 기본적으로 전체 특성 수 n의 $\sqrt{n}$개 특성을 선택
- 무작위성을 더하여 트리를 다양하게 만들고, 편향을 늘리는 대신 분산을 낮춤.

In [33]:
#최대 16개의 리프 노드를 갖는 트리 500개로 랜덤 포레스트를 훈련
from sklearn.ensemble import RandomForestClassifier

rnd_clf = RandomForestClassifier(n_estimators=500, max_leaf_nodes=16,
                                  n_jobs=-1, random_state=42)
rnd_clf.fit(X_train, y_train)

y_pred_rf = rnd_clf.predict(X_test)

In [34]:
#BaggingClassifier는 RandomForestClassifier와 거의 동일하게 구성한 것
bag_clf = BaggingClassifier(
    DecisionTreeClassifier(max_features="sqrt", max_leaf_nodes=16),
    n_estimators=500, n_jobs=-1, random_state=42)

### **7.4.1 엑스트라 트리**

- 최적의 임곗값을 찾는 대신 후보 특성으로 랜덤 분할을 만든 후 최상의 분할을 선택
- 결정 트리에 적용하려면 `splitter="random"`으로 지정
- 이런 트리의 앙상블을 **익스트림 랜덤 트리**, 줄여서 **엑스트라 트리**
- 편향이 늘어나는 대신 분산이 낮아짐. 최적의 임곗값을 찾는 작업을 줄여 훈련 속도가 빠름
- 분류: `ExtraTreesClassifier`, 회귀: `ExtraTreesRegressor`
- 랜덤 포레스트 클래스와 달리 `bootstrap`의 기본값이 `False`임
- 랜덤 포레스트와 엑스트라 트리 중 어느 것이 더 나은지는 교차 검증으로 비교

### **7.4.2 특성 중요도**

- 각 특성을 사용한 노드가 평균적으로 불순도를 얼마나 감소시키는지 측정함.
- 노드와 연관된 훈련 샘플 수를 가중치로 사용하는 가중치 평균임.
- 훈련 후 중요도의 전체 합이 1이 되도록 정규화함.
- 결과는 `feature_importances_`에 저장됨.


In [35]:
#iris 데이터셋으로 각 특성의 중요도를 확인
from sklearn.datasets import load_iris
iris = load_iris(as_frame=True)
rnd_clf = RandomForestClassifier(n_estimators=500, random_state=42)
rnd_clf.fit(iris.data, iris.target)
for score, name in zip(rnd_clf.feature_importances_, iris.data.columns):
    print(round(score, 2), name)
# 결과 - 꽃잎의 길이와 너비가 꽃받침의 길이와 너비보다 중요하게 나타남.
# MNIST에서도 각 픽셀의 중요도를 확인할 수 있음. (281쪽,그림 7-6)
#특성을 선택할 때 어떤 특성이 중요한지 빠르게 확인할 수 있음.

0.11 sepal length (cm)
0.02 sepal width (cm)
0.44 petal length (cm)
0.42 petal width (cm)


- MNIST에서도 각 픽셀의 중요도를 확인할 수 있음. (281쪽,그림 7-6)

- 특성을 선택할 때 어떤 특성이 중요한지 빠르게 확인할 수 있음.

# **7.5 부스팅**

**부스팅**: 약한 학습기를 여러 개 연결하여 강한 학습기를 만드는 앙상블 방법 - 앞의 모델을 보완하면서 예측기를 순차적으로 학습시킴.

대표적인 방법 => **AdaBoost**와 **그레이디언트 부스팅**

### **7.5.1 AdaBoost**

- 이전 모델이 잘못 분류한 훈련 샘플의 가중치를 상대적으로 높임.
- 다음 예측기는 업데이트된 가중치로 훈련하므로 학습하기 어려운 샘플에 더 맞춰짐.
- 훈련과 예측, 가중치 업데이트를 반복하면서 예측기를 추가
- 경사 하강법은 한 예측기의 모델 파라미터를 조정하지만, AdaBoost는 앙상블에 예측기를 추가
- 예측기마다 가중치가 적용된 훈련 세트에서의 정확도에 따라 다른 가중치 부여ㅕ
- 이전 예측기의 훈련과 평가가 끝나야 다음 예측기를 훈련할 수 있으므로 훈련을 병렬화X

**AdaBoost의 오류율과 예측기 가중치**

각 샘플의 가중치 $w^{(i)}$는 처음에 $\frac{1}{m}$으로 초기화함.


$$
r_j=
\frac{
\displaystyle\sum_{\substack{i=1\\\hat{y}_j^{(i)}\ne y^{(i)}}}^{m}w^{(i)}
}{
\displaystyle\sum_{i=1}^{m}w^{(i)}
}
$$

$\hat{y}_j^{(i)}$는 $i$번째 샘플에 대한 $j$번째 예측기의 예측

$$
\alpha_j=\eta\log\frac{1-r_j}{r_j}
$$

- $\eta$: 학습률 하이퍼파라미터. 기본값 1
- 정확할수록 예측기의 가중치가 높아짐.
- 랜덤 추측이면 가중치가 0에 가까워지고, 랜덤 추측보다 정확도가 낮으면 음수

**가중치 업데이트 규칙**

$$
w^{(i)}\leftarrow
\begin{cases}
w^{(i)} & \hat{y}_j^{(i)}=y^{(i)}\text{일 때}\\
w^{(i)}\exp(\alpha_j) & \hat{y}_j^{(i)}\ne y^{(i)}\text{일 때}
\end{cases}
\qquad i=1,2,\ldots,m
$$

- 잘못 분류된 샘플의 가중치를 높임.
- 업데이트 후 모든 가중치를 $\sum_{i=1}^{m}w^{(i)}$로 나누어 정규화
- 업데이트된 가중치로 다음 예측기를 훈련함
- 지정한 예측기 수에 도달하거나 완벽한 예측기가 만들어지면 중지

**AdaBoost 예측**

$$
\hat{y}(\mathbf{x})=
\underset{k}{\operatorname{argmax}}
\sum_{\substack{j=1\\\hat{y}_j(\mathbf{x})=k}}^{N}\alpha_j
$$

- $N$은 예측기 수. 각 클래스에 투표한 예측기들의 가중치를 더해 가장 큰 합을 가진 클래스를 예측

**SAMME와 SAMME.R 설명**

- **SAMME**: AdaBoost의 다중 클래스 버전. 클래스가 두 개면 AdaBoost와 동일
- **SAMME.R**: 예측값 대신 클래스 확률을 사용함. 예측기가 `predict_proba()`를 제공할 때 사용한다고 설명
- 회귀를 위한 `AdaBoostRegressor`도 제공, 기본 추정기는 깊이가 3인 `DecisionTreeRegressor`


In [36]:
#결정 트리는 `max_depth=1`로, 결정 노드 하나와 리프 노드 두 개로 구성
from sklearn.ensemble import AdaBoostClassifier

ada_clf = AdaBoostClassifier(
    DecisionTreeClassifier(max_depth=1), n_estimators=30,
    learning_rate=0.5, random_state=42)
ada_clf.fit(X_train, y_train)

AdaBoostClassifier(estimator=DecisionTreeClassifier(max_depth=1),
                   learning_rate=0.5, n_estimators=30, random_state=42)

-> AdaBoost가 훈련 세트에 과대적합되면 추정기 수를 줄이거나 추정기의 규제를 더 강하기

### **7.5.2 그레이디언트 부스팅**

- 이전까지의 오차를 보정하도록 예측기를 순차적으로 추가
- AdaBoost처럼 샘플 가중치를 수정하는 대신, 이전 예측기가 만든 **잔여 오차**에 새로운 예측기를 학습
- 결정 트리를 사용하는 회귀 방법을 **그레이디언트 트리 부스팅** 또는 **그레이디언트 부스티드 회귀 트리(GBRT)**라고 함.

In [37]:
#잡음이 섞인 2차 방정식 데이터셋을 만들고 첫 번째 결정 트리를 훈련
import numpy as np
from sklearn.tree import DecisionTreeRegressor
np.random.seed(42)
X = np.random.rand(100, 1) - 0.5
y = 3 * X[:, 0] ** 2 + 0.05 * np.random.randn(100) # y = 3x² + 가우스_잡음
tree_reg1 = DecisionTreeRegressor(max_depth=2, random_state=42)
tree_reg1.fit(X, y)

DecisionTreeRegressor(max_depth=2, random_state=42)

In [38]:
#첫 번째 예측기의 잔여 오차에 두 번째 결정 트리를 훈련
y2 = y - tree_reg1.predict(X)
tree_reg2 = DecisionTreeRegressor(max_depth=2, random_state=43)
tree_reg2.fit(X, y2)

DecisionTreeRegressor(max_depth=2, random_state=43)

In [39]:
#두 번째 예측기까지 보정한 뒤 남은 잔여 오차에 세 번째 결정 트리를 훈련
y3 = y2 - tree_reg2.predict(X)
tree_reg3 = DecisionTreeRegressor(max_depth=2, random_state=44)
tree_reg3.fit(X, y3)

DecisionTreeRegressor(max_depth=2, random_state=44)

-> 새로운 샘플의 예측은 세 트리의 예측을 더하여 맍듬

In [40]:
X_new = np.array([[-0.4], [0.], [0.5]])
sum(tree.predict(X_new) for tree in (tree_reg1, tree_reg2, tree_reg3))

array([0.49484029, 0.04021166, 0.75026781])

**GradientBoostingRegressor**

- GBRT 앙상블을 간단하게 훈련할 수 있음.
- 분류에는 `GradientBoostingClassifier`를 사용
- `n_estimators`: 트리 수
- `max_depth`, `min_samples_leaf`: 결정 트리의 성장을 제어

In [41]:
#앞에서 만든 것과 같은 앙상블
from sklearn.ensemble import GradientBoostingRegressor

gbrt = GradientBoostingRegressor(max_depth=2, n_estimators=3,
                                  learning_rate=1.0, random_state=42)
gbrt.fit(X, y)

GradientBoostingRegressor(learning_rate=1.0, max_depth=2, n_estimators=3,
                          random_state=42)

**학습률과 축소**

- `learning_rate`는 각 트리의 기여도를 조절
- 낮게 설정하면 더 많은 트리가 필요하지만 일반적으로 예측 성능이 좋아짐. ->  **축소**라는 규제 방법
- 트리가 부족하면 과소적합되고, 너무 많이 추가하면 과대적합될 수 있음.


**조기 종료**

- 최적의 트리 수는 `GridSearchCV`, `RandomizedSearchCV`로 찾을 수 있음.
- 더 간단한 방법은 `n_iter_no_change`를 설정하는 것
- `n_iter_no_change=10`: 마지막 10번의 반복에서 진전이 없으면 트리 추가를 중지
- 너무 낮게 설정하면 과소적합, 너무 높게 설정하면 과대적합될 수 있음.

In [42]:
gbrt_best = GradientBoostingRegressor(
    max_depth=2, learning_rate=0.05, n_estimators=500,
    n_iter_no_change=10, random_state=42)
gbrt_best.fit(X, y)

GradientBoostingRegressor(learning_rate=0.05, max_depth=2, n_estimators=500,
                          n_iter_no_change=10, random_state=42)

In [43]:
gbrt_best.n_estimators_
#결과 -트리 수를 500개로 설정했지만 조기 종료로 실제 추정기 수는 92개

92

- `n_iter_no_change`를 설정하면 `fit()`이 훈련 세트를 훈련용과 검증용으로 나눔
- `validation_fraction`: 검증 세트의 비율. 기본값은 10%
- `tol`: 무시할 수 있는 최대 성능 향상. 이때 기본값은 0.0001


**확률적 그레이디언트 부스팅**

- `subsample`은 각 트리의 훈련에 사용할 샘플 비율을 지정
- `subsample=0.25`이면 각 트리는 랜덤으로 선택한 25%의 훈련 샘플로 학습
- 편향이 높아지는 대신 분산이 낮아지고 훈련 속도가 빨라짐.

### **7.5.3 히스토그램 기반 그레이디언트 부스팅**

**히스토그램 기반 그레이디언트 부스팅(HGB)**: 대규모 데이터셋에 최적화된 GBRT 구현

- 입력 특성을 구간으로 나누어 정수로 대체
- `max_bins`: 구간의 개수. 교재의 기본값은 255이며, 이보다 높게 설정할 수 X
- 평가할 임곗값의 수를 줄이고, 각 트리를 학습할 때 특성을 정렬할 필요가 없어 훈련 속도를 높임
- 구간 분할은 규제처럼 작동-> 과대적합을 줄일 수도 있지만 과소적합을 유발할 수도 있음.

In [53]:
import pandas as pd
import numpy as np
from sklearn.pipeline import make_pipeline
from sklearn.compose import make_column_transformer
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.preprocessing import OrdinalEncoder

# 캘리포니아 주택 데이터셋 로드
housing_df = pd.read_csv('/content/sample_data/california_housing_train.csv')

# 파이프라인의 OrdinalEncoder 작동을 위해 가상의 범주형 특성 'ocean_proximity' 추가
np.random.seed(42)
categories = ['<1H OCEAN', 'INLAND', 'NEAR OCEAN', 'NEAR BAY', 'ISLAND']
housing_df['ocean_proximity'] = np.random.choice(categories, size=len(housing_df))

# 특성(X)과 타깃(y) 분리
housing = housing_df.drop("median_house_value", axis=1)
housing_labels = housing_df["median_house_value"]

# 파이프라인 구성 및 훈련
hgb_reg = make_pipeline(
    make_column_transformer((OrdinalEncoder(), ["ocean_proximity"]),
                             remainder="passthrough"),
    HistGradientBoostingRegressor(categorical_features=[0], random_state=42)
)
hgb_reg.fit(housing, housing_labels)

/usr/local/lib/python3.13/dist-packages/sklearn/compose/_column_transformer.py:1667: FutureWarning: 
The format of the columns of the 'remainder' transformer in ColumnTransformer.transformers_ will change in version 1.7 to match the format of the other transformers.
At the moment the remainder columns are stored as indices (of type int). With the same ColumnTransformer configuration, in the future they will be stored as column names (of type str).
To use the new behavior now and suppress this warning, use ColumnTransformer(force_int_remainder_cols=False).

  warnings.warn(


Pipeline(steps=[('columntransformer',
                 ColumnTransformer(remainder='passthrough',
                                   transformers=[('ordinalencoder',
                                                  OrdinalEncoder(),
                                                  ['ocean_proximity'])])),
                ('histgradientboostingregressor',
                 HistGradientBoostingRegressor(categorical_features=[0],
                                               random_state=42))])

- 누락된 값을 채우거나 스케일을 조정하거나 원-핫 인코딩을 처리X 괜춘
- 하이퍼파라미터 튜닝 없이  47,600의 RMSE 얻음.
- **XGBoost**, **CatBoost**, **LightGBM**.
- **텐서플로 랜덤 포레스트**는 랜덤 포레스트·엑스트라 트리·GBRT 등의 최적화된 구현을 제공


# **7.6 스태킹**

**스태킹**: 예측을 취합하는 간단한 함수 대신, 예측을 취합하는 모델을 훈련하는 방법(stacked generalization 줄임말)

- 마지막 예측기 => **블렌더** 또는 **메타 학습기**
- 블렌더가 기본 예측기들의 예측을 입력으로 받아 최종 예측을 만듬

**블렌더 훈련 과정**

1. 각 예측기에 `cross_val_predict()`를 사용하여 원본 훈련 샘플에 대한 표본 외 예측을 얻음
2. 이 예측을 블렌더의 입력 특성으로 사용하고, 타깃은 원본 훈련 세트에서 복사
3. 교재의 회귀 예시에서는 예측기당 하나의 입력 특성이 만들어짐
4. 블렌더를 훈련한 뒤 기본 예측기들을 전체 원본 훈련 세트로 다시 훈련

-> 블렌더 계층을 여러 개 쌓을 수도 있지만 훈련 시간과 시스템 복잡성이 증가

In [54]:
from sklearn.ensemble import StackingClassifier

stacking_clf = StackingClassifier(
    estimators=[
        ('lr', LogisticRegression(random_state=42)),
        ('rf', RandomForestClassifier(random_state=42)),
        ('svc', SVC(probability=True, random_state=42))
    ],
    final_estimator=RandomForestClassifier(random_state=43),
    cv=5 # 교차 검증 폴드 개수
)
stacking_clf.fit(X_train, y_train)

StackingClassifier(cv=5,
                   estimators=[('lr', LogisticRegression(random_state=42)),
                               ('rf', RandomForestClassifier(random_state=42)),
                               ('svc', SVC(probability=True, random_state=42))],
                   final_estimator=RandomForestClassifier(random_state=43))

**스태킹 분류기의 예측과 최종 예측기**

- 기본 예측기에 `predict_proba()`가 있으면 그거 사용 없으면 `decision_function()`, 마지막으로 `predict()`를 사용
- 최종 예측기를 지정하지 X이면 `StackingClassifier`는 `LogisticRegression`, `StackingRegressor`는 `RidgeCV`를 사용


-> 랜덤포레스트·AdaBoost·GBRT가 표 형식 데이터에 유용하고, 투표 기반 분류기와 스태킹이 시스템의 성능을 높이는 데 도움